In [1]:
import gc
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Load transaction data
# ---------------------------------------------------------

db = pd.read_csv(
    "../data/raw/train_transaction.csv"
)

print("Transaction:", db.shape)


# ---------------------------------------------------------
# 2. Sort TRANSACTION data before merging
# ---------------------------------------------------------

db = db.sort_values(
    "TransactionDT",
    kind="mergesort"
).reset_index(drop=True)

print("Transaction sorted successfully.")


# ---------------------------------------------------------
# 3. Load identity data
# ---------------------------------------------------------

identity = pd.read_csv(
    "../data/raw/train_identity.csv"
)

print("Identity:", identity.shape)


# ---------------------------------------------------------
# 4. Merge
# ---------------------------------------------------------

data_590k = db.merge(
    identity,
    on="TransactionID",
    how="left",
    sort=False
)

print("Merged data:", data_590k.shape)


# ---------------------------------------------------------
# 5. Free unnecessary objects
# ---------------------------------------------------------

del db
del identity

gc.collect()

print("Memory cleanup completed.")

Transaction: (590540, 394)
Transaction sorted successfully.
Identity: (144233, 41)
Merged data: (590540, 434)
Memory cleanup completed.


In [2]:
data_590k["has_identity"] = (
    data_590k["id_01"].notna()
).astype(np.int8)

In [3]:
data_590k["log_TransactionAmt"] = np.log1p(
    data_590k["TransactionAmt"]
)

In [4]:
data_590k["TransactionHour"] = (
    (data_590k["TransactionDT"] // 3600) % 24
).astype(np.int8)

In [5]:
data_590k["TransactionDay"] = (
    data_590k["TransactionDT"] // (24 * 3600)
).astype(np.int32)

In [6]:
data_590k["time_since_previous"] = (
    data_590k["TransactionDT"].diff()
).fillna(0)

In [7]:
data_590k["amount_change"] = (
    data_590k["TransactionAmt"].diff()
).fillna(0)

In [8]:
data_590k["abs_amount_change"] = (
    data_590k["amount_change"].abs()
)

In [9]:
data_590k["card1_frequency"] = (
    data_590k.groupby("card1").cumcount()
)

In [10]:
data_590k["card1_previous_amount_sum"] = (
    data_590k.groupby("card1")["TransactionAmt"]
    .cumsum()
    - data_590k["TransactionAmt"]
)

In [11]:
data_590k["card1_previous_count"] = (
    data_590k.groupby("card1").cumcount()
)

In [12]:
data_590k["card1_avg_previous_amount"] = (
    data_590k["card1_previous_amount_sum"]
    /
    data_590k["card1_previous_count"].replace(
        0,
        np.nan
    )
)

In [13]:
data_590k["amount_vs_card_avg"] = (
    data_590k["TransactionAmt"]
    /
    (data_590k["card1_avg_previous_amount"] + 1e-6)
)

In [14]:
data_590k["card1_avg_previous_amount"] = (
    data_590k["card1_avg_previous_amount"]
    .fillna(data_590k["TransactionAmt"])
)

In [15]:
data_590k["amount_vs_card_avg"] = (
    data_590k["TransactionAmt"]
    /
    (data_590k["card1_avg_previous_amount"] + 1e-6)
)

In [16]:
data_590k["recent_card_transactions"] = (
    data_590k.groupby("card1")["TransactionDT"]
    .transform(
        lambda x: x.rolling(
            window=10,
            min_periods=1
        ).count()
    )
)

In [17]:
data_590k = data_590k.drop(
    columns=[
        "card1_previous_amount_sum",
        "card1_previous_count"
    ]
)

In [18]:
behavioral_features = [
    "log_TransactionAmt",
    "TransactionHour",
    "TransactionDay",
    "has_identity",
    "time_since_previous",
    "amount_change",
    "abs_amount_change",
    "card1_frequency",
    "card1_avg_previous_amount",
    "amount_vs_card_avg",
    "recent_card_transactions"
]

print("Behavioral features:")
print(behavioral_features)

Behavioral features:
['log_TransactionAmt', 'TransactionHour', 'TransactionDay', 'has_identity', 'time_since_previous', 'amount_change', 'abs_amount_change', 'card1_frequency', 'card1_avg_previous_amount', 'amount_vs_card_avg', 'recent_card_transactions']


In [19]:
data_590k[behavioral_features].describe().T

,count,mean,std,min,25%,50%,75%,max
log_TransactionAmt,590540.0,4.382960,0.937183,0.223943,3.791459,4.245190,4.836282,10.371564
TransactionHour,590540.0,13.861923,7.607152,0.000000,6.000000,16.000000,20.000000,23.000000
TransactionDay,590540.0,84.729199,53.437277,1.000000,35.000000,84.000000,130.000000,182.000000
has_identity,590540.0,0.244239,0.429636,0.000000,0.000000,0.000000,0.000000,1.000000
time_since_previous,590540.0,26.627715,59.299048,0.000000,5.000000,13.000000,29.000000,4138.000000
amount_change,590540.0,0.000358,336.014408,-31918.782000,-60.000000,0.000000,60.111250,31878.391000
abs_amount_change,590540.0,147.227132,302.042741,0.000000,23.788500,60.000000,149.950000,31918.782000
card1_frequency,590540.0,1263.907732,2258.936861,0.000000,43.000000,312.000000,1382.000000,14931.000000
card1_avg_previous_amount,590540.0,129.238058,86.808837,0.615000,90.935284,115.576885,148.474057,4517.710000
amount_vs_card_avg,590540.0,1.085659,1.713462,0.002896,0.397015,0.674718,1.140773,145.944802


In [20]:
print(
    "Missing values:",
    data_590k[behavioral_features].isnull().sum().sum()
)

Missing values: 0


In [21]:
print(
    "Infinite values:",
    np.isinf(
        data_590k[behavioral_features]
        .select_dtypes(include=np.number)
    ).sum().sum()
)

Infinite values: 0


In [22]:
import os
os.makedirs(
    "../data/processed",
    exist_ok=True
)
data_590k.to_parquet(
    "../data/processed/razorshield_engineered_590k.parquet",
    index=False
)
print("Saved successfully!")
print("Shape:", data_590k.shape)

Saved successfully!
Shape: (590540, 445)


In [23]:
data_590k.groupby("isFraud")[
    "amount_vs_card_avg"
].agg(
    ["count", "mean", "median"]
)

,count,mean,median
isFraud,,,
0,569877,1.079650,0.670164
1,20663,1.251382,0.826401


In [24]:
data_590k.groupby("isFraud")[
    "card1_frequency"
].agg(
    ["count", "mean", "median"]
)

,count,mean,median
isFraud,,,
0,569877,1264.703161,308.0
1,20663,1241.970140,414.0


In [25]:
data_590k.groupby("has_identity")[
    "isFraud"
].agg(
    ["count", "sum", "mean"]
)

,count,sum,mean
has_identity,,,
0,446307,9345,0.020939
1,144233,11318,0.078470


In [26]:
data_590k.groupby("TransactionHour")[
    "isFraud"
].agg(
    ["count", "sum", "mean"]
).sort_values(
    "mean",
    ascending=False
)

,count,sum,mean
TransactionHour,,,
7,3704,393,0.106102
8,2591,241,0.093014
9,2479,223,0.089956
6,6007,467,0.077743
5,9701,682,0.070302
10,3627,193,0.053212
4,14839,770,0.051890
11,6827,265,0.038816
3,20802,797,0.038314


In [27]:
import numpy as np
import pandas as pd

data = pd.read_parquet(
    "../data/processed/razorshield_engineered_590k.parquet"
)

print("Data shape:", data.shape)

Data shape: (590540, 445)


In [28]:
X = data.drop(
    columns=["isFraud", "TransactionID"]
)

y = data["isFraud"]

print("X:", X.shape)
print("y:", y.shape)

X: (590540, 443)
y: (590540,)


In [29]:
split_1 = int(len(X) * 0.70)
split_2 = int(len(X) * 0.85)

X_train = X.iloc[:split_1].copy()
X_val = X.iloc[split_1:split_2].copy()
X_test = X.iloc[split_2:].copy()

y_train = y.iloc[:split_1].copy()
y_val = y.iloc[split_1:split_2].copy()
y_test = y.iloc[split_2:].copy()

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (413378, 443)
Validation: (88581, 443)
Test: (88581, 443)


In [30]:
print("Train fraud rate:", y_train.mean())
print("Validation fraud rate:", y_val.mean())
print("Test fraud rate:", y_test.mean())

print("\nFraud counts:")
print("Train:", y_train.sum())
print("Validation:", y_val.sum())
print("Test:", y_test.sum())

Train fraud rate: 0.03516878014795175
Validation fraud rate: 0.03434145019812375
Test fraud rate: 0.03480430340592226

Fraud counts:
Train: 14538
Validation: 3042
Test: 3083


In [31]:
categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Categorical columns:", len(categorical_columns))
print(categorical_columns)

Categorical columns: 31
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


In [32]:
sparse_columns = [
    col
    for col in X_train.columns
    if X_train[col].isnull().mean() > 0.90
]

print("Sparse columns:", len(sparse_columns))
print(sparse_columns)

Sparse columns: 12
['dist2', 'D7', 'id_07', 'id_08', 'id_18', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27']


In [33]:
X_train = X_train.drop(columns=sparse_columns)
X_val = X_val.drop(columns=sparse_columns)
X_test = X_test.drop(columns=sparse_columns)

In [34]:
categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_columns = X_train.select_dtypes(
    include=["int64", "float64", "float32", "int32"]
).columns.tolist()

print("Categorical:", len(categorical_columns))
print("Numerical:", len(numerical_columns))

Categorical: 29
Numerical: 400


In [35]:
train_medians = X_train[numerical_columns].median()

X_train[numerical_columns] = X_train[numerical_columns].fillna(
    train_medians
)

X_val[numerical_columns] = X_val[numerical_columns].fillna(
    train_medians
)

X_test[numerical_columns] = X_test[numerical_columns].fillna(
    train_medians
)

In [36]:
X_train[categorical_columns] = (
    X_train[categorical_columns].fillna("Unknown")
)

X_val[categorical_columns] = (
    X_val[categorical_columns].fillna("Unknown")
)

X_test[categorical_columns] = (
    X_test[categorical_columns].fillna("Unknown")
)

In [37]:
high_cardinality_columns = [
    col
    for col in categorical_columns
    if X_train[col].nunique() > 100
]

print(
    "High-cardinality columns:",
    high_cardinality_columns
)

High-cardinality columns: ['id_31', 'id_33', 'DeviceInfo']


In [38]:
X_train = X_train.drop(
    columns=high_cardinality_columns
)

X_val = X_val.drop(
    columns=high_cardinality_columns
)

X_test = X_test.drop(
    columns=high_cardinality_columns
)

In [39]:
categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

X_train = pd.get_dummies(
    X_train,
    columns=categorical_columns
)

X_val = pd.get_dummies(
    X_val,
    columns=categorical_columns
)

X_test = pd.get_dummies(
    X_test,
    columns=categorical_columns
)

In [40]:
X_val = X_val.reindex(
    columns=X_train.columns,
    fill_value=0
)

X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

In [41]:
X_train = X_train.astype(np.float32)
X_val = X_val.astype(np.float32)
X_test = X_test.astype(np.float32)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (413378, 673)
Validation: (88581, 673)
Test: (88581, 673)


In [42]:
print("Missing values:")

print(
    "Train:",
    X_train.isnull().sum().sum()
)

print(
    "Validation:",
    X_val.isnull().sum().sum()
)

print(
    "Test:",
    X_test.isnull().sum().sum()
)

Missing values:
Train: 0
Validation: 0
Test: 0


In [43]:
from xgboost import XGBClassifier

negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive

print(
    "Scale positive weight:",
    scale_pos_weight
)

Scale positive weight: 27.434310083918007


In [78]:
xgb_v2 = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,

    scale_pos_weight=scale_pos_weight,

    objective="binary:logistic",
    eval_metric="aucpr",

    tree_method="hist",

    random_state=42,
    n_jobs=-1
)

In [79]:
xgb_v2.fit(
    X_train,
    y_train
)
print("XGBoost V2 training completed.")

XGBoost V2 training completed.


In [80]:
xgb_v2_val_prob = xgb_v2.predict_proba(
    X_val
)[:, 1]

In [81]:
xgb_v2_val_pred = (
    xgb_v2_val_prob >= 0.5
).astype(int)

In [82]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

def evaluate_model(
    name,
    y_true,
    y_pred,
    y_prob
):

    print("=" * 60)
    print(name)
    print("=" * 60)

    print(
        f"Accuracy : {accuracy_score(y_true, y_pred):.4f}"
    )

    print(
        f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}"
    )

    print(
        f"Recall   : {recall_score(y_true, y_pred, zero_division=0):.4f}"
    )

    print(
        f"F1       : {f1_score(y_true, y_pred, zero_division=0):.4f}"
    )

    print(
        f"ROC-AUC  : {roc_auc_score(y_true, y_prob):.4f}"
    )

    print(
        f"PR-AUC   : {average_precision_score(y_true, y_prob):.4f}"
    )

    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            y_true,
            y_pred
        )
    )

In [83]:
evaluate_model(
    "RazorShield XGBoost V2 - Validation",
    y_val,
    xgb_v2_val_pred,
    xgb_v2_val_prob
)

RazorShield XGBoost V2 - Validation
Accuracy : 0.9349
Precision: 0.3009
Recall   : 0.6765
F1       : 0.4166
ROC-AUC  : 0.9151
PR-AUC   : 0.5508

Confusion Matrix:
[[80758  4781]
 [  984  2058]]


In [84]:
threshold_results = []

for threshold in np.arange(
    0.05,
    0.96,
    0.05
):

    pred = (
        xgb_v2_val_prob >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_val,
            pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_val,
            pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_val,
            pred,
            zero_division=0
        )
    })

threshold_df_v2 = pd.DataFrame(
    threshold_results
)

threshold_df_v2

,threshold,precision,recall,f1
0,0.05,0.044459,0.990138,0.085097
1,0.10,0.061959,0.965155,0.116443
2,0.15,0.084149,0.934911,0.154402
3,0.20,0.109704,0.899737,0.195563
4,0.25,0.135372,0.857002,0.233812
5,0.30,0.162821,0.814267,0.271378
6,0.35,0.192117,0.778764,0.308203
7,0.40,0.225470,0.745562,0.346233
8,0.45,0.260435,0.711703,0.381330
9,0.50,0.300921,0.676529,0.416557


In [85]:
best_row_v2 = threshold_df_v2.loc[
    threshold_df_v2["f1"].idxmax()
]

best_threshold_v2 = best_row_v2["threshold"]

print("Best threshold:", best_threshold_v2)
print("Precision:", best_row_v2["precision"])
print("Recall:", best_row_v2["recall"])
print("F1:", best_row_v2["f1"])

Best threshold: 0.7500000000000001
Precision: 0.5879275653923541
Recall: 0.480276134122288
F1: 0.5286774018454858


In [86]:
xgb_v2_test_prob = xgb_v2.predict_proba(
    X_test
)[:, 1]

xgb_v2_test_pred = (
    xgb_v2_test_prob >= best_threshold_v2
).astype(int)

In [87]:
evaluate_model(
    "RazorShield XGBoost V2 - FINAL TEST",
    y_test,
    xgb_v2_test_pred,
    xgb_v2_test_prob
)

RazorShield XGBoost V2 - FINAL TEST
Accuracy : 0.9676
Precision: 0.5412
Recall   : 0.4535
F1       : 0.4935
ROC-AUC  : 0.9017
PR-AUC   : 0.5152

Confusion Matrix:
[[84313  1185]
 [ 1685  1398]]


In [88]:
xgb_v2_train_prob = xgb_v2.predict_proba(X_train)[:, 1]

xgb_v2_train_pred = (
    xgb_v2_train_prob >= best_threshold_v2
).astype(int)

In [89]:
xgb_v2_train_prob = xgb_v2.predict_proba(X_train)[:, 1]

xgb_v2_train_pred = (
    xgb_v2_train_prob >= best_threshold_v2
).astype(int)

In [90]:
# =========================================================
# TRAINING SET EVALUATION
# =========================================================

xgb_v2_train_prob = xgb_v2.predict_proba(X_train)[:, 1]

xgb_v2_train_pred = (
    xgb_v2_train_prob >= best_threshold_v2
).astype(int)

evaluate_model(
    "RazorShield XGBoost V2 - TRAIN",
    y_train,
    xgb_v2_train_pred,
    xgb_v2_train_prob
)

RazorShield XGBoost V2 - TRAIN
Accuracy : 0.9784
Precision: 0.6753
Recall   : 0.7421
F1       : 0.7071
ROC-AUC  : 0.9741
PR-AUC   : 0.7869

Confusion Matrix:
[[393653   5187]
 [  3750  10788]]


In [91]:
evaluate_model(
    "RazorShield XGBoost V2 - FINAL TEST",
    y_train,
    xgb_v2_train_pred,
    xgb_v2_train_prob
)

RazorShield XGBoost V2 - FINAL TEST
Accuracy : 0.9784
Precision: 0.6753
Recall   : 0.7421
F1       : 0.7071
ROC-AUC  : 0.9741
PR-AUC   : 0.7869

Confusion Matrix:
[[393653   5187]
 [  3750  10788]]


In [92]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(
    xgb_v2,
    "../models/razorshield_xgboost_v2.pkl"
)

print("XGBoost V2 model saved successfully.")

XGBoost V2 model saved successfully.


In [93]:
joblib.dump(
    best_threshold_v2,
    "../models/razorshield_xgboost_v2_threshold.pkl"
)

print("Threshold saved:", best_threshold_v2)

Threshold saved: 0.7500000000000001


In [94]:
feature_columns = X_train.columns.tolist()

joblib.dump(
    feature_columns,
    "../models/razorshield_xgboost_v2_features.pkl"
)

print("Feature columns saved:", len(feature_columns))

Feature columns saved: 673


In [95]:
import joblib

loaded_model = joblib.load(
    "../models/razorshield_xgboost_v2.pkl"
)

loaded_threshold = joblib.load(
    "../models/razorshield_xgboost_v2_threshold.pkl"
)

loaded_features = joblib.load(
    "../models/razorshield_xgboost_v2_features.pkl"
)

print("Model loaded:", type(loaded_model))
print("Threshold:", loaded_threshold)
print("Features:", len(loaded_features))

Model loaded: <class 'xgboost.sklearn.XGBClassifier'>
Threshold: 0.7500000000000001
Features: 673
